## Step 1 - Install Packages

In [22]:
!pip install scikit-learn==1.6.0 streamlit==1.30.0 pandas==2.0.3 numpy==1.26.4 matplotlib==3.8.2 seaborn plotly joblib

  Using cached scikit_learn-1.6.0-cp313-cp313-win_amd64.whl.metadata (15 kB)
  Using cached streamlit-1.30.0-py2.py3-none-any.whl.metadata (8.2 kB)
  Using cached pandas-2.0.3.tar.gz (5.3 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [23 lines of output]
      Traceback (most recent call last):
        File "C:\Users\Fatima\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 353, in <module>
          main()
          ~~~~^^
        File "C:\Users\Fatima\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 335, in main
          json_out['return_val'] = hook(**hook_input['kwargs'])
                                   ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
        File "C:\Users\Fatima\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 118, in get_requires_for_build_wheel
          return hook(config_settings)
        File "C:\Users\Fatima\AppData\Local\Temp\pip-build-env-fpxb1zx4\overlay\Lib\site-p

## Step 2 - Import Liabraries

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import joblib

from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Step 3 - Load the Dataset

In [23]:
df = pd.read_csv("global_ecommerce_extreme_regression_dataset.csv")


## Step 4 - Exporatory Data Analysis(EDA) - Before Cleaning

In [11]:
df.head()
df.tail()
df.info()
df.describe()
df.isnull().sum()
print("Duplicates:", df.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 360500 entries, 0 to 360499
Data columns (total 89 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   customer_id                360500 non-null  int64  
 1   age                        354788 non-null  str    
 2   country_code               360500 non-null  str    
 3   region                     360500 non-null  str    
 4   signup_date                360500 non-null  str    
 5   loyalty_score              360500 non-null  str    
 6   discount_usage_rate        360500 non-null  float64
 7   email_open_rate            316334 non-null  float64
 8   avg_review_score           360500 non-null  float64
 9   customer_tier              360500 non-null  str    
 10  credit_limit               360500 non-null  str    
 11  referral_code              360500 non-null  str    
 12  device_preference          360500 non-null  str    
 13  traffic_source             360500 non-nu

In [13]:
target_col = 'future_customer_spend_90d'
s = pd.to_numeric(
    df[target_col].astype(str).str.replace(r'[^0-9.-]', '', regex=True),
    errors='coerce'
).dropna()

print("Min:", s.min(), "Median:", s.median(), "95th pct:", s.quantile(0.95), "Max:", s.max())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 1. Zoomed in to the bulk of the data (e.g., up to 95th percentile)
cap = s.quantile(0.95)
sns.histplot(s[s <= cap], bins=40, kde=True, ax=axes[0])
axes[0].set_title(f"Spend Distribution (<= 95th pct: {cap:,.0f})")

# 2. Log scale (for positive values) to reveal heavy tail across orders of magnitude
sns.histplot(s[s > 0], bins=40, log_scale=True, ax=axes[1])
axes[1].set_title("Spend Distribution (Log X-Axis, > 0)")

plt.tight_layout()
plt.savefig('results/figures/target_distribution_zoomed.png', dpi=150)
plt.show()

Min: 0.0 Median: 137.835 95th pct: 574.1004999999999 Max: 72490.81


## Step 5 — Copy data, remove noise columns, clean issues, and impute missing values

In [29]:
# Step 5: Copy data, remove noise columns, clean issues, and impute missing values
df_new = df.copy()

# 1. Standardize column names (remove spaces, lowercase)
df_new.columns = df_new.columns.str.strip().str.lower()

# 2. Remove noise_feature_* columns
noise_cols = [c for c in df_new.columns if c.startswith('noise_feature')]
df_new = df_new.drop(columns=noise_cols)

# 3. Convert all "missing" tokens to real NaN and standardize text
missing_values = [
    '?', '??', 'n/a', 'na', 'null', 'none',
    'unknown', 'missing', 'error', '-', '--', '', ' '
]
df_new = df_new.replace(missing_values, np.nan)

# Strip spaces and lowercase all text columns
object_cols = df_new.select_dtypes(include='object').columns
for col in object_cols:
    df_new[col] = df_new[col].astype(str).str.strip().str.lower()

# astype(str) turns NaN into the text 'nan', so convert it back
df_new = df_new.replace(['nan', 'none', ''], np.nan)

# 4. Standardize country codes, device preference, and traffic source
country_mapping = {
    'us': 'usa',
    'u.s.': 'usa',
    'u.s.a.': 'usa',
    'united states': 'usa',
    'united states of america': 'usa',
    'u.k.': 'uk',
    'great britain': 'uk',
    'britain': 'uk',
    'u.a.e.': 'uae',
    'united arab emirates': 'uae'
}
if 'country_code' in df_new.columns:
    df_new['country_code'] = df_new['country_code'].replace(country_mapping)

if 'device_preference' in df_new.columns:
    df_new['device_preference'] = df_new['device_preference'].replace({
        'mobile': 'mobile',
        'desktop': 'desktop',
        'tablet': 'tablet'
    })

if 'traffic_source' in df_new.columns:
    traffic_mapping = {
        'organic': 'organic',
        'email': 'email',
        'affiliate': 'affiliate',
        'social': 'social',
        'ads': 'ads'
    }
    df_new['traffic_source'] = df_new['traffic_source'].replace(traffic_mapping)

# 5. Convert text values to numbers
if 'is_active' in df_new.columns:
    df_new['is_active'] = df_new['is_active'].replace({
        'yes': 1, 'y': 1, 'true': 1,
        'no': 0, 'n': 0, 'false': 0
    })

if 'loyalty_score' in df_new.columns:
    df_new['loyalty_score'] = df_new['loyalty_score'].replace({
        'low': 0.33,
        'medium': 0.66,
        'high': 1.00
    })

# 'many' is not a real number, so treat it as missing
if 'avg_pages_viewed' in df_new.columns:
    df_new['avg_pages_viewed'] = df_new['avg_pages_viewed'].replace({
        'many': np.nan
    })

# 6. Remove currency symbols and commas from money columns
money_columns = [
    'credit_limit',
    'avg_order_value',
    'lifetime_spend',
    'future_customer_spend_90d'
]
for col in money_columns:
    if col in df_new.columns:
        df_new[col] = (
            df_new[col]
            .astype(str)
            .str.replace('$', '', regex=False)
            .str.replace(',', '', regex=False)
            .str.strip()
        )

# Convert all numeric columns to numbers (invalid values become NaN)
numeric_columns = [
    'age',
    'loyalty_score',
    'discount_usage',
    'discount_usage_rate',
    'email_open_rate',
    'avg_review_score',
    'credit_limit',
    'avg_order_value',
    'lifetime_spend',
    'is_active',
    'marketing_id',
    'avg_session_duration',
    'avg_cart_additions',
    'avg_pages_viewed',
    'bounce_rate',
    'num_sessions_last_30d',
    'num_transactions',
    'average_income',
    'urban_ratio',
    'internet_penetration',
    'future_customer_spend_90d'
]
for col in numeric_columns:
    if col in df_new.columns:
        df_new[col] = pd.to_numeric(df_new[col], errors='coerce')

# 7. Drop rows with missing target, then remove duplicate rows
df_new = df_new.dropna(subset=['future_customer_spend_90d'])
df_new = df_new.drop_duplicates().reset_index(drop=True)

# 8. Fill ALL remaining missing values
# Numeric columns -> replace with the column mean
num_cols = df_new.select_dtypes(include=[np.number]).columns
df_new[num_cols] = df_new[num_cols].fillna(df_new[num_cols].mean())

# Text columns -> replace with the most frequent value (mean doesn't work for text)
cat_cols = df_new.select_dtypes(include=['object', 'category']).columns
for col in cat_cols:
    if df_new[col].isnull().any():
        df_new[col] = df_new[col].fillna(df_new[col].mode()[0])

# Safety net: if a column was entirely empty, fill any leftovers with 0
df_new = df_new.fillna(0)

# 9. Check the results
print("Removed noise columns:", len(noise_cols))
print("Cleaned shape:", df_new.shape)
print("Remaining duplicates:", df_new.duplicated().sum())
print("Remaining missing values:\n", df_new.isnull().sum()[df_new.isnull().sum() > 0])

# 10. Save the cleaned dataset
df_new.to_csv('cleaned_global_ecommerce_dataset.csv', index=False)
df_new.head()

Removed noise columns: 60
Cleaned shape: (350000, 29)
Remaining duplicates: 0
Remaining missing values:
 Series([], dtype: int64)


,customer_id,age,country_code,region,signup_date,loyalty_score,discount_usage_rate,email_open_rate,avg_review_score,customer_tier,...,avg_cart_additions,avg_pages_viewed,bounce_rate,num_sessions_last_30d,num_transactions,average_income,urban_ratio,internet_penetration,region_tier,future_customer_spend_90d
0,182994,43.0,au,region_95,2022-01-26,0.3461,0.313795,0.618761,5.066010,bronze,...,5,10.0,0.231956,8,3,30729.701037,0.480949,0.618679,tier3,114.18
1,229191,52.0,in,region_62,2023-07-14,0.0038,0.323821,0.607257,3.314194,diamond,...,0,5.0,0.464669,4,5,37873.300520,0.679706,0.588395,tier2,87.21
2,242964,27.0,usa,region_145,2022-06-22,0.2633,0.541580,0.259109,3.542250,gold,...,2,7.0,0.324568,3,3,75636.457436,0.538404,0.510991,tier1,132.28
3,306869,59.0,de,region_120,2021-12-15,0.2405,0.168623,0.534019,4.078036,platinum,...,1,11.0,0.206675,5,5,75820.638941,0.322021,0.531922,tier3,160.50
4,342915,63.0,fr,region_31,2023-06-07,0.2479,0.236642,0.432682,3.834070,silver,...,2,7.0,0.181683,7,5,97045.563690,0.692288,0.566294,tier3,574.72


## Step 6 - Separate input features and target

In [32]:
df_new = df.copy()
df_new = df_new.drop_duplicates().reset_index(drop=True)

def clean_currency_series(col):
    return pd.to_numeric(
        col.astype(str).str.replace(r"[^0-9.-]", "", regex=True),
        errors="coerce",
    )

# Clean target and drop rows with missing target
df_new[target_col] = clean_currency_series(df_new[target_col])
df_new = df_new.dropna(subset=[target_col]).reset_index(drop=True)

# Identify candidate feature columns (update num_features / cat_features explicitly if desired)
candidate_cols = [c for c in df_new.columns if c != target_col]

# Separate object/category columns from numeric/currency columns
cat_features = [
    c for c in candidate_cols
    if df_new[c].dtype == "object" or str(df_new[c].dtype) == "category"
]

# Check if any object columns are actually currency strings rather than categories
currency_like_cols = []
true_cat_features = []
for c in cat_features:
    parsed = clean_currency_series(df_new[c])
    non_null_orig = df_new[c].dropna()
    non_null_parsed = parsed.dropna()
    if len(non_null_orig) > 0 and (len(non_null_parsed) / len(non_null_orig)) > 0.8:
        currency_like_cols.append(c)
    else:
        true_cat_features.append(c)

num_features = [
    c for c in candidate_cols if c not in true_cat_features
]

for c in num_features:
    df_new[c] = clean_currency_series(df_new[c])
    df_new[c] = df_new[c].fillna(df_new[c].median())

for c in true_cat_features:
    df_new[c] = df_new[c].astype(str).str.strip().str.lower()
    df_new.loc[df_new[c].isin(["nan", "none", "null", ""]), c] = np.nan
    mode_vals = df_new[c].mode(dropna=True)
    fill_val = mode_vals.iloc[0] if len(mode_vals) > 0 else "missing"
    df_new[c] = df_new[c].fillna(fill_val)

print("Numeric features:", num_features)
print("Categorical features:", true_cat_features)


# separate columns
X = df_new[num_features + true_cat_features]
y = df_new[target_col]

Numeric features: ['customer_id', 'age', 'country_code', 'region', 'signup_date', 'loyalty_score', 'discount_usage_rate', 'email_open_rate', 'avg_review_score', 'customer_tier', 'credit_limit', 'referral_code', 'device_preference', 'traffic_source', 'avg_order_value', 'lifetime_spend', 'is_active', 'marketing_id', 'avg_session_duration', 'avg_cart_additions', 'avg_pages_viewed', 'bounce_rate', 'num_sessions_last_30d', 'num_transactions', 'average_income', 'urban_ratio', 'internet_penetration', 'region_tier', 'noise_feature_0', 'noise_feature_1', 'noise_feature_2', 'noise_feature_3', 'noise_feature_4', 'noise_feature_5', 'noise_feature_6', 'noise_feature_7', 'noise_feature_8', 'noise_feature_9', 'noise_feature_10', 'noise_feature_11', 'noise_feature_12', 'noise_feature_13', 'noise_feature_14', 'noise_feature_15', 'noise_feature_16', 'noise_feature_17', 'noise_feature_18', 'noise_feature_19', 'noise_feature_20', 'noise_feature_21', 'noise_feature_22', 'noise_feature_23', 'noise_feature_2

## Step 7 - Train / Test split

In [43]:
# Step 7: Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Step 7.1: Sample only the training data
sample_size = min(50000, len(X_train))

X_train_sample, _, y_train_sample, _ = train_test_split(
    X_train,
    y_train,
    train_size=sample_size,
    random_state=42
)

# Use the sample for training
X_train_model = X_train_sample
y_train_model = y_train_sample

print("Training rows used:", len(X_train))
print("Testing rows kept:", len(X_test))

Training rows used: 280000
Testing rows kept: 70001


## Step 8 — Create preprocessing pipelines


In [44]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)),
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, true_cat_features),
])

## Step 9 - Regression models and evaluation

In [45]:
def print_reg_metrics(
    name,
    model,
    X_tr,
    y_tr,
    X_te,
    y_te
):
    train_predictions = model.predict(X_tr)
    test_predictions = model.predict(X_te)

    train_mae = mean_absolute_error(
        y_tr,
        train_predictions
    )

    train_rmse = np.sqrt(
        mean_squared_error(
            y_tr,
            train_predictions
        )
    )

    train_r2 = r2_score(
        y_tr,
        train_predictions
    )

    test_mae = mean_absolute_error(
        y_te,
        test_predictions
    )

    test_rmse = np.sqrt(
        mean_squared_error(
            y_te,
            test_predictions
        )
    )

    test_r2 = r2_score(
        y_te,
        test_predictions
    )

    print(f"=== {name} ===")

    print(
        f"Train -> MAE: {train_mae:.2f} | "
        f"RMSE: {train_rmse:.2f} | "
        f"R2: {train_r2:.4f}"
    )

    print(
        f"Test  -> MAE: {test_mae:.2f} | "
        f"RMSE: {test_rmse:.2f} | "
        f"R2: {test_r2:.4f}\n"
    )

#  Train Ridge Regression
ridge_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "regressor",
            Ridge(alpha=1.0)
        )
    ]
)

ridge_model.fit(
    X_train_model,
    y_train_model
)

print_reg_metrics(
    "Ridge Regression",
    ridge_model,
    X_train_model,
    y_train_model,
    X_test,
    y_test
)

rf_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=50,
                max_depth=15,
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

rf_model.fit(
    X_train_model,
    y_train_model
)

print_reg_metrics(
    "Random Forest Regressor",
    rf_model,
    X_train_model,
    y_train_model,
    X_test,
    y_test
)

=== Ridge Regression ===
Train -> MAE: 245.67 | RMSE: 1224.11 | R2: 0.0063
Test  -> MAE: 253.96 | RMSE: 1288.53 | R2: 0.0024

=== Random Forest Regressor ===
Train -> MAE: 231.37 | RMSE: 992.55 | R2: 0.3467
Test  -> MAE: 271.74 | RMSE: 1315.39 | R2: -0.0397



In [46]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Generate predictions
test_predictions = rf_model.predict(X_test)

# Define the high-value spending threshold using training data only
high_value_threshold = y_train_model.quantile(0.75)

# Convert actual and predicted spending into high-value labels
actual_high_value = (
    y_test >= high_value_threshold
).astype(int)

predicted_high_value = (
    test_predictions >= high_value_threshold
).astype(int)

# Calculate classification metrics
accuracy = accuracy_score(
    actual_high_value,
    predicted_high_value
)

precision = precision_score(
    actual_high_value,
    predicted_high_value,
    zero_division=0
)

recall = recall_score(
    actual_high_value,
    predicted_high_value,
    zero_division=0
)

f1 = f1_score(
    actual_high_value,
    predicted_high_value,
    zero_division=0
)

print("High-Value Customer Classification Metrics")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

High-Value Customer Classification Metrics
Accuracy:  0.6286
Precision: 0.3454
Recall:    0.5288
F1-score:  0.4179


## Step 10 - Cross Validation

In [54]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

# Use the sampled training data only
x_train = X_train_model
y_train = y_train_model

print("Rows used for cross-validation:", len(x_train))
print("Rows reserved for final testing:", len(X_test))

# Use a smaller Random Forest during cross-validation
rf_model_cv = Pipeline([
    ("preprocessor", preprocessor),
    (
        "regressor",
        RandomForestRegressor(
            n_estimators=30,
            max_depth=10,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=1
        )
    )
])

# Use three folds for faster testing
kfold = KFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

cv_results = cross_validate(
    estimator=rf_model_cv,
    X=x_train,
    y=y_train,
    cv=kfold,
    scoring=[
        "r2",
        "neg_mean_absolute_error",
        "neg_root_mean_squared_error"
    ],
    return_train_score=False,
    n_jobs=-1
)

cv_results

Rows used for cross-validation: 50000
Rows reserved for final testing: 70001


{'fit_time': array([56.21439695, 55.64237595, 56.11639762]),
 'score_time': array([0.08628583, 0.07708049, 0.07865548]),
 'test_r2': array([-0.01724947, -0.02823413, -0.05151554]),
 'test_neg_mean_absolute_error': array([-246.53890716, -270.59688021, -250.07087859]),
 'test_neg_root_mean_squared_error': array([-1178.74669028, -1396.64395427, -1151.23223565])}

## Step 11 - Hyperparameter tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

# Hyperparameter values to test
param_grid = {
    "regressor__n_estimators": [30, 50],
    "regressor__max_depth": [10, 15],
    "regressor__min_samples_leaf": [2, 4]
}

# Grid search using 3-fold cross-validation
grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=3,
    scoring="r2",
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

# Train using the sampled training data only
grid_search.fit(
    x_train,
    y_train
)

# Display the best hyperparameters
print("Best parameters:")
print(grid_search.best_params_)

print(
    "Best cross-validation R2:",
    round(grid_search.best_score_, 4)
)

# Select the best tuned model
best_rf_model = grid_search.best_estimator_

# Evaluate the tuned model on the untouched test data
print_reg_metrics(
    "Tuned Random Forest Regressor",
    best_rf_model,
    x_train,
    y_train,
    x_test,
    y_test
)

Fitting 3 folds for each of 8 candidates, totalling 24 fits


In [ ]:
final_pipeline = grid_search.best_estimator_
joblib.dump(final_pipeline, MODEL_PATH)
print("Saved pipeline to:", MODEL_PATH)